# M05D · Web Scraping Fundamentals for Biological Data
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M05_Databases_and_Scraping/M05D_Web_Scraping_Fundamentals.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M02A (Text Files), M05A (NCBI Entrez)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Evaluate**: Determine when web scraping is ethically and methodologically appropriate compared to REST APIs and bulk FTP downloads.
- **Audit**: Inspect and programmatically evaluate `robots.txt` permissions using Python's `urllib.robotparser.RobotFileParser`.
- **Navigate**: Traverse the HTML Document Object Model (DOM) using BeautifulSoup (`find`, `find_all`, CSS selectors) to isolate target biological data.
- **Extract**: Transform unstructured and tabular HTML structures (`<table>`, `<tr>`, `<td>`) into tidy pandas DataFrames.
- **Simulate**: Parse authentic offline biological HTML pages (`data/html/`) representing PubMed search results, UniProt entries, and biomarker tables.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Sending HTTP requests and handling responses with `requests` ([`M05C`](file:///modules/M05_Databases_and_Scraping/M05C_Best_Practices_API_Keys.ipynb)).
- Parsing text and manipulating strings with Python ([`M01A`](file:///modules/M01_Python_Basics/M01A_Core_Python.ipynb)).
- Basic HTML vocabulary (tags like `<div>`, `<p>`, `<a>`, `<table>`, and attributes like `class` and `id`).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M05D"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, re, time, io
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 5D.1 The Scraping Decision Framework: API First

### 4.1.1 Theory & Biological Context

Web scraping is the automated extraction of data from websites designed for human browser consumption.

### The Biological Data Access Hierarchy:
Always follow the **API-First Rule**:

```
1. Bulk Archive / FTP Downloads (Fastest, least load, e.g. UniProt complete proteome, Ensembl FASTA dumps)
      │
      ▼ (if specific subset needed)
2. Documented REST / GraphQL APIs (Structured JSON/TSV, rate limits defined, e.g. NCBI E-utils, RCSB PDB)
      │
      ▼ (if no API exists and bulk data unavailable)
3. Web Scraping (Fragile, dependent on HTML layout, requires strict politeness)
```

### When is Scraping Justified?
- Niche biological databases lacking programmatic APIs (e.g. specialized academic lab servers, rare plant registries, regional clinical trial portals).
- Aggregating data across legacy web forms.
- Extracting supplementary tables embedded directly in journal HTML articles.

### 4.1.2 Worked Example

In [ ]:
# 5D.1 Worked Example: Data Access Decision Helper
def evaluate_data_source(has_bulk_download: bool, has_rest_api: bool, allows_scraping: bool) -> str:
    """Determines the most responsible biological data acquisition path."""
    if has_bulk_download:
        return "PRIORITY 1: Download bulk gzip/tarball archive via FTP/HTTPS."
    elif has_rest_api:
        return "PRIORITY 2: Query documented REST API with structured JSON/TSV parameters."
    elif allows_scraping:
        return "PRIORITY 3: Scrape politely (verify robots.txt, identify User-Agent, throttle requests)."
    else:
        return "REJECT: Scraping disallowed by Terms of Service and no API available. Contact data maintainers."

print("GEO expression matrix:", evaluate_data_source(has_bulk_download=True, has_rest_api=True, allows_scraping=True))
print("UniProt queries:      ", evaluate_data_source(has_bulk_download=False, has_rest_api=True, allows_scraping=True))
print("Legacy lab portal:    ", evaluate_data_source(has_bulk_download=False, has_rest_api=False, allows_scraping=True))

### 4.1.3 ✏️ Try It: Access Strategy Evaluator

Write a function `recommend_access_strategy(portal_metadata: dict) -> str`:
- If `portal_metadata.get("api_available")`: return `"Use REST API"`
- Elif `portal_metadata.get("ftp_available")`: return `"Use FTP Download"`
- Elif `portal_metadata.get("scraping_permitted")`: return `"Scrape Politely"`
- Else return `"Do Not Access"`

In [ ]:
def recommend_access_strategy(portal_metadata: dict) -> str:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
assert recommend_access_strategy({"api_available": True}) == "Use REST API"
assert recommend_access_strategy({"api_available": False, "ftp_available": True}) == "Use FTP Download"
assert recommend_access_strategy({"api_available": False, "ftp_available": False, "scraping_permitted": True}) == "Scrape Politely"
assert recommend_access_strategy({}) == "Do Not Access"
print("Section 5D.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Scraping a website that provides a public API**: Scraping HTML when a REST API exists violates provider terms of service, wastes bandwidth, and produces fragile code.
- **Ignoring site Terms of Service**: Many commercial and institutional health portals explicitly forbid automated crawling; always inspect terms before writing code.

## 5D.2 Legal & Ethical Considerations: robots.txt & Politeness

### 4.2.1 Theory & Biological Context

Automated scraping places real computational load on target servers. As scientists, we must adhere to high ethical standards.

### The Five Pillars of Polite Scraping:
1. **Respect `robots.txt`**: Web servers publish a `robots.txt` file at their domain root (e.g. `https://example.org/robots.txt`) declaring which bots are permitted to visit specific URL paths.
2. **Declare Identity via `User-Agent`**:
   - Never masquerade as a standard browser (like Chrome or Firefox) to deceive the server.
   - Set a descriptive User-Agent header with contact details:
     `User-Agent: PCB-Course-Bot/1.0 (student@iiitd.ac.in; +https://github.com/meprateekpaul/ProgrammingCB)`
3. **Pace Requests (`time.sleep`)**: Add delays ($1-3$ seconds) between consecutive page requests.
4. **Never Scrape Personal / Protected Health Information**: Scraping patient records or clinical data without institutional review board (IRB) approval violates HIPAA, GDPR, and data protection statutes.
5. **Cache Locally**: Save scraped HTML locally so subsequent analyses do not hit the server again.

### 4.2.2 Worked Example

In [ ]:
# 5D.2 Worked Example: Programmatic robots.txt Auditing with RobotFileParser
def check_robots_permission(robots_url: str, user_agent: str, target_url: str) -> bool:
    """Reads and audits robots.txt permissions for a specific user agent and target URL."""
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        can_fetch = rp.can_fetch(user_agent, target_url)
        return can_fetch
    except Exception as e:
        print(f"Notice: robots.txt could not be fetched ({e}), defaulting to cautious policy.")
        return False

# Demonstrate with sample simulated rules
sample_robots_txt = """User-agent: *
Disallow: /admin/
Disallow: /private/
Disallow: /api/internal/
Allow: /data/
"""

rp_sim = RobotFileParser()
rp_sim.parse(sample_robots_txt.splitlines())

print("Can fetch /data/biomarkers? ", rp_sim.can_fetch("MyBot", "https://example.org/data/biomarkers"))
print("Can fetch /admin/users?     ", rp_sim.can_fetch("MyBot", "https://example.org/admin/users"))
print("Can fetch /api/internal/id? ", rp_sim.can_fetch("MyBot", "https://example.org/api/internal/id"))

### 4.2.3 ✏️ Try It: Robots Text Auditor

Write a function `audit_robots_text(robots_content: str, agent: str, test_url: str) -> bool`:
- Takes raw `robots_content` string, parses with `RobotFileParser.parse()`.
- Tests if `agent` is permitted to access `test_url`.
- Returns `True` if allowed, `False` if disallowed.

In [ ]:
def audit_robots_text(robots_content: str, agent: str, test_url: str) -> bool:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
mock_rules = """User-agent: BioScraper
Disallow: /restricted/
Allow: /public/
"""
assert audit_robots_text(mock_rules, "BioScraper", "https://site.org/public/genes")
assert not audit_robots_text(mock_rules, "BioScraper", "https://site.org/restricted/keys")
print("Section 5D.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Ignoring Disallow directives in robots.txt**: Disregarding `Disallow` rules can trigger automatic IP bans and potential legal liability under Computer Fraud and Abuse statutes.
- **Using generic or blank User-Agent headers**: Requests with empty or default python-requests User-Agents are immediately rejected by web application firewalls (Cloudflare, AWS WAF).

## 5D.3 HTTP Basics: Methods, Headers & Status Codes

### 4.3.1 Theory & Biological Context

Web scraping relies on the Hypertext Transfer Protocol (HTTP).

### Anatomy of an HTTP Request:
1. **Method**: `GET` (retrieving content) or `POST` (submitting form data).
2. **URL**: Scheme (`https`), Domain (`www.ncbi.nlm.nih.gov`), Path (`/gene/7157`), Query parameters (`?format=json`).
3. **Headers**: Key-value pairs providing client metadata (`User-Agent`, `Accept`, `Authorization`).
4. **Status Codes**:
   - `200 OK`: Request succeeded.
   - `301 / 302 Redirect`: Resource moved.
   - `400 Bad Request`: Malformed syntax.
   - `403 Forbidden`: Access denied (often bot detection).
   - `404 Not Found`: Resource does not exist.
   - `429 Too Many Requests`: Rate limit exceeded.
   - `500 Internal Server Error`: Target server crashed.

### 4.3.2 Worked Example

In [ ]:
# 5D.3 Worked Example: Configuring Custom Headers and Inspecting Responses
headers = {
    "User-Agent": "PCB-Course-Bot/1.0 (Computational Biology Class; student@iiitd.ac.in)",
    "Accept": "text/html,application/xhtml+xml"
}

# Inspect response status codes and headers
print(f"Configured Custom Headers:\n  User-Agent: {headers['User-Agent']}")

### 4.3.3 ✏️ Try It: Custom Header Builder

Write a function `build_custom_headers(bot_name: str, email: str) -> dict[str, str]`:
- Returns a dictionary with `'User-Agent': f'{bot_name}/1.0 ({email})'` and `'Accept': 'text/html'`.

In [ ]:
def build_custom_headers(bot_name: str, email: str) -> dict[str, str]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
h = build_custom_headers("ResearchBot", "pi@lab.org")
assert "ResearchBot/1.0" in h["User-Agent"]
assert "pi@lab.org" in h["User-Agent"]
assert h["Accept"] == "text/html"
print("Section 5D.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Omitting timeout on requests.get**: Always pass `timeout=10` to avoid hangs on dead sockets.
- **Not checking response.status_code**: Never parse `response.text` without verifying `response.status_code == 200` or calling `response.raise_for_status()`.

## 5D.4 HTML Structure & The Document Object Model (DOM)

### 4.4.1 Theory & Biological Context

Web pages are structured as HTML (HyperText Markup Language) documents represented in memory as a hierarchical tree called the **Document Object Model (DOM)**.

### DOM Tree Architecture:
```
<html>
  ├── <head> (<title>)
  └── <body>
        ├── <header>
        └── <div class="container">
              ├── <h1> (Title)
              └── <table id="biomarkers">
                    ├── <thead> (<th> headers)
                    └── <tbody> (<tr> rows -> <td> cells)
```

### Why Regular Expressions Fail on HTML:
Regular expressions parse regular languages. HTML contains arbitrarily nested tags, multiline attributes, and unescaped entities:
```html
<div class="main">
   <div class="nested">
      <p>Target gene: TP53</p>
   </div>
</div>
```
A regex matching `<div>(.*?)</div>` matches the inner closing tag prematurely!
**Always use a dedicated HTML parser (BeautifulSoup).**

### 4.4.2 Worked Example

In [ ]:
# 5D.4 Worked Example: Inspecting HTML DOM Structure with BeautifulSoup
raw_html_snippet = """
<div class="gene-card" id="gene-tp53" data-entrez="7157">
    <h2 class="gene-title">Tumor Protein P53</h2>
    <span class="organism">Homo sapiens</span>
    <ul class="aliases">
        <li>P53</li>
        <li>BCC7</li>
        <li>LFS1</li>
    </ul>
</div>
"""

soup_snippet = BeautifulSoup(raw_html_snippet, "html.parser")
card = soup_snippet.find("div", class_="gene-card")

print(f"Tag Name:   {card.name}")
print(f"Class:      {card['class']}")
print(f"Data-ID:    {card['data-entrez']}")
print(f"Title Text: {card.find('h2').get_text()}")
print(f"Aliases:    {[li.get_text() for li in card.find_all('li')]}")

### 4.4.3 ✏️ Try It: Tag Metadata Extractor

Write a function `extract_tag_metadata(html_str: str) -> dict`:
- Parse `html_str` with BeautifulSoup.
- Find the first `<a>` tag.
- Extract and return a dictionary with:
  - `'text'`: `tag.get_text(strip=True)`
  - `'href'`: `tag.get('href', '')`
  - `'target'`: `tag.get('target', '_self')`

In [ ]:
def extract_tag_metadata(html_str: str) -> dict:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
snippet = '<a href="/gene/7157" target="_blank">View TP53</a>'
res = extract_tag_metadata(snippet)
assert res["text"] == "View TP53"
assert res["href"] == "/gene/7157"
assert res["target"] == "_blank"
print("Section 5D.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Using regex on HTML**: Writing regexes like `<a href='(.*?)'>(.*?)</a>` breaks on attributes in different order or newlines. Always use BeautifulSoup.
- **Using tag['attr'] when attribute might be missing**: If an attribute is absent, `tag['href']` raises `KeyError`. Use `tag.get('href', '')` for safe access.

## 5D.5 Parsing HTML with BeautifulSoup & CSS Selectors

### 4.5.1 Theory & Biological Context

BeautifulSoup provides two primary paradigms for locating elements:

### 1. Tag Navigation Methods:
- `soup.find('tag', attrs={'class': 'val'})`: Returns the **first** matching element.
- `soup.find_all('tag', class_='val')`: Returns a **list** of all matching elements.
- Navigating the tree: `.parent`, `.children`, `.find_next_sibling()`.

### 2. CSS Selectors (`soup.select()`):
Mirrors the selector syntax used by web browsers and frontend CSS stylesheets:
- Tag selector: `soup.select("p")`
- Class selector: `soup.select(".docsum-title")`
- ID selector: `soup.select("#protein-name")`
- Child combinator: `soup.select("div.search-results > article")`
- Attribute selector: `soup.select("article[data-pmid]")`

### 4.5.2 Worked Example

In [ ]:
# 5D.5 Worked Example: Parsing Offline PubMed Search Results
pubmed_html_path = DATA_DIR / "html" / "pubmed_sample.html"
html_content = pubmed_html_path.read_text(encoding="utf-8")

soup = BeautifulSoup(html_content, "html.parser")

# Use CSS selectors to find all article cards
articles = soup.select("article.docsum-journal-citation")
print(f"Found {len(articles)} PubMed article cards in sample HTML:")

for art in articles:
    pmid = art.get("data-pmid", "N/A")
    title_tag = art.select_one("a.docsum-title")
    title = title_tag.get_text(strip=True) if title_tag else "N/A"
    authors_tag = art.select_one("span.docsum-authors")
    authors = authors_tag.get_text(strip=True) if authors_tag else "N/A"
    journal_tag = art.select_one("span.docsum-journal")
    journal = journal_tag.get_text(strip=True) if journal_tag else "N/A"
    
    print(f"  PMID: {pmid}")
    print(f"    Title:   {title[:70]}...")
    print(f"    Authors: {authors}")
    print(f"    Journal: {journal}\n")

### 4.5.3 ✏️ Try It: PMID Card Extractor

Write a function `extract_all_pmids(soup_obj: BeautifulSoup) -> list[str]`:
- Finds all `article` elements with attribute `data-pmid`.
- Returns a list of the PMID strings.

In [ ]:
def extract_all_pmids(soup_obj: BeautifulSoup) -> list[str]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
pmids_extracted = extract_all_pmids(soup)
assert len(pmids_extracted) >= 3
assert "38123456" in pmids_extracted
print("Section 5D.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Confusing find() with find_all()**: `find()` returns a single `Tag` object, while `find_all()` returns a list of tags. Calling `.get_text()` directly on `find_all()` raises `AttributeError`.
- **Forgetting strip=True in get_text()**: Web pages contain indentation whitespace and newlines around text. Calling `.get_text(strip=True)` removes surrounding whitespace.

## 5D.6 Extracting Tabular Data: Manual Parsing & pd.read_html

### 4.6.1 Theory & Biological Context

HTML tables are one of the most common representations of biological data (e.g. differential expression results, biomarker panels, clinical cohorts).

### Structure of an HTML Table:
- `<table>`: Table wrapper.
- `<thead>`: Header section containing `<th>` (table header) cells.
- `<tbody>`: Body section containing `<tr>` (table row) elements.
- `<td>`: Table data cells inside rows.

### Two Parsing Approaches:
1. **Manual BeautifulSoup traversal**: Parse `th` for column names, iterate over `tr` rows, extract `td` text, construct list of dicts. Full control over data types, cleaning, and custom attributes.
2. **`pandas.read_html()`**: High-level wrapper that parses all `<table>` elements in an HTML string directly into DataFrames.

### 4.6.2 Worked Example

In [ ]:
# 5D.6 Worked Example: Parsing Offline Biomarker Table
table_html_path = DATA_DIR / "html" / "table_sample.html"
table_html = table_html_path.read_text(encoding="utf-8")

# Method 1: Manual BeautifulSoup Parsing
soup_tab = BeautifulSoup(table_html, "html.parser")
table_el = soup_tab.find("table", id="biomarkers")

# Extract headers
headers = [th.get_text(strip=True) for th in table_el.find("thead").find_all("th")]

# Extract body rows
rows = []
for tr in table_el.find("tbody").find_all("tr"):
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    if cells:
        row_dict = dict(zip(headers, cells))
        rows.append(row_dict)

df_biomarkers = pd.DataFrame(rows)
# Convert numeric columns
df_biomarkers["Expression_FoldChange"] = pd.to_numeric(df_biomarkers["Expression_FoldChange"])
df_biomarkers["P_Value"] = pd.to_numeric(df_biomarkers["P_Value"])

print("Manually Parsed Biomarker DataFrame:")
print(df_biomarkers)

# Method 2: pandas.read_html shortcut
dfs = pd.read_html(io.StringIO(table_html))
print(f"\npandas.read_html extracted {len(dfs)} table(s):")
print(dfs[0].head(2))

### 4.6.3 ✏️ Try It: Significant Biomarker Filter

Write a function `filter_significant_biomarkers(df: pd.DataFrame, min_fc: float = 2.0, max_p: float = 0.05) -> pd.DataFrame`:
- Filters `df` for rows where `Expression_FoldChange >= min_fc` and `P_Value <= max_p`.
- Returns the filtered DataFrame.

In [ ]:
def filter_significant_biomarkers(df: pd.DataFrame, min_fc: float = 2.0, max_p: float = 0.05) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
df_sig = filter_significant_biomarkers(df_biomarkers, min_fc=2.0, max_p=0.05)
assert len(df_sig) > 0
assert all(df_sig["Expression_FoldChange"] >= 2.0)
assert all(df_sig["P_Value"] <= 0.05)
print("Section 5D.6 passed ✔")

### 4.6.4 ⚠️ Common Mistakes

- **Leaving numeric columns as strings**: HTML parsers extract strings (`'2.45'`). Trying to filter without converting via `pd.to_numeric()` performs lexicographic string comparison (`'10'` < `'2'`), producing erroneous results.
- **Assuming read_html always works**: `pd.read_html()` fails on poorly formed tables or tables generated dynamically with JavaScript. Manual BeautifulSoup traversal is the reliable foundation.

## 5D.7 Offline Parsing of Biological HTML Datasets

### 4.7.1 Theory & Biological Context

To ensure complete reproducibility and protect student evaluations from live web server downtime, our course includes offline fixtures in `data/html/`:
1. `pubmed_sample.html`: Formatted literature query output.
2. `uniprot_sample.html`: Curated protein record entry.
3. `table_sample.html`: Structured clinical biomarker matrix.

### 4.7.2 Worked Example

In [ ]:
# 5D.7 Worked Example: Parsing Offline UniProt Entry
uniprot_html_path = DATA_DIR / "html" / "uniprot_sample.html"
u_html = uniprot_html_path.read_text(encoding="utf-8")

u_soup = BeautifulSoup(u_html, "html.parser")

prot_name = u_soup.select_one("#protein-name").get_text(strip=True)
gene_name = u_soup.select_one(".gene-name").get_text(strip=True)
organism = u_soup.select_one(".organism").get_text(strip=True)
func_text = u_soup.select_one(".description").get_text(strip=True)

print(f"Protein:   {prot_name}")
print(f"Gene:      {gene_name}")
print(f"Organism:  {organism}")
print(f"Function:  {func_text[:90]}...")

### 4.7.3 ✏️ Try It: UniProt HTML Record Parser

Write a function `parse_uniprot_html(html_str: str) -> dict`:
- Parse `html_str` using BeautifulSoup.
- Extract `'protein_name'`, `'gene_name'`, `'organism'`, and `'function'`.
- Return a dictionary with these 4 keys.

In [ ]:
def parse_uniprot_html(html_str: str) -> dict:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
u_parsed = parse_uniprot_html(u_html)
assert u_parsed["gene_name"] == "TP53"
assert "Homo sapiens" in u_parsed["organism"]
assert len(u_parsed["function"]) > 20
print("Section 5D.7 passed ✔")

### 4.7.4 ⚠️ Common Mistakes

- **Assuming elements exist without checking None**: If an element is missing, `soup.select_one(...)` returns `None`. Calling `.get_text()` on `None` raises `AttributeError`.
- **Hardcoding index positions in HTML**: Relying on positional indexing (e.g. `soup.find_all('div')[4]`) breaks whenever minor styling divs are added to the page.

### Practice Exercise 5D.1 (★) — Automated robots.txt Path Auditor

**Problem**: Write a function `audit_crawling_permissions(robots_txt: str, paths_to_test: list[str], bot_user_agent: str = "PCB-Bot") -> dict[str, bool]`:
- Takes a raw `robots_txt` content string.
- Parses it with `RobotFileParser`.
- For each relative path in `paths_to_test` (e.g. `"/data/genes"`), tests if `bot_user_agent` is allowed to fetch `"https://example.org" + path`.
- Returns a dictionary mapping `{path: can_fetch_bool}`.


- **Input**: Robots.txt string, list of paths, and user agent string
- **Expected Output**: Dictionary mapping each path to True/False permission

In [ ]:
def audit_crawling_permissions(robots_txt: str, paths_to_test: list[str], bot_user_agent: str = "PCB-Bot") -> dict[str, bool]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
rules = """User-agent: *
Disallow: /admin/
Disallow: /cgi-bin/
Disallow: /temp/
Allow: /data/
"""
audited = audit_crawling_permissions(rules, ["/data/genes", "/admin/keys", "/cgi-bin/blast", "/about"])
assert audited["/data/genes"] is True
assert audited["/admin/keys"] is False
assert audited["/cgi-bin/blast"] is False
assert audited["/about"] is True
print("Ex 5D.1 passed ✔")

### Practice Exercise 5D.2 (★) — Extracting Overexpressed Biomarkers from HTML Table

**Problem**: Write a function `extract_upregulated_biomarkers(html_path: Path, min_fold_change: float = 2.0) -> pd.DataFrame`:
- Read HTML file at `html_path` (e.g. `data/html/table_sample.html`).
- Parse the table using BeautifulSoup.
- Extract rows where `Expression_FoldChange >= min_fold_change`.
- Return a pandas DataFrame with columns `['Symbol', 'Name', 'Chromosome', 'Expression_FoldChange', 'P_Value']` sorted by `Expression_FoldChange` descending.


- **Input**: Path to table_sample.html and minimum fold change float
- **Expected Output**: DataFrame of upregulated biomarkers sorted descending

In [ ]:
def extract_upregulated_biomarkers(html_path: Path, min_fold_change: float = 2.0) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
df_up = extract_upregulated_biomarkers(DATA_DIR / "html" / "table_sample.html", min_fold_change=2.0)
assert len(df_up) > 0
assert df_up["Expression_FoldChange"].iloc[0] >= df_up["Expression_FoldChange"].iloc[-1]
assert all(df_up["Expression_FoldChange"] >= 2.0)
print("Ex 5D.2 passed ✔")

### Practice Exercise 5D.3 (★★) — PubMed HTML Search Results Parser

**Problem**: Write a function `parse_pubmed_html_results(html_path: Path) -> pd.DataFrame`:
- Read `pubmed_sample.html`.
- Parse all `<article>` tags in the results.
- Extract:
  - `'pmid'`: integer PMID from `data-pmid` attribute
  - `'title'`: article title from `a.docsum-title`
  - `'first_author'`: first author name from `span.docsum-authors` (first name before comma or whole text if no comma)
  - `'journal'`: journal name
- Return a DataFrame with these columns.


- **Input**: Path to pubmed_sample.html
- **Expected Output**: DataFrame of parsed articles with columns [pmid, title, first_author, journal]

In [ ]:
def parse_pubmed_html_results(html_path: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
df_pub = parse_pubmed_html_results(DATA_DIR / "html" / "pubmed_sample.html")
assert len(df_pub) >= 3
assert "pmid" in df_pub.columns and "title" in df_pub.columns
assert df_pub["pmid"].dtype in [int, np.int64, np.int32]
print("Ex 5D.3 passed ✔")

### Practice Exercise 5D.4 (★★) — Extracting Feature Tables from UniProt HTML

**Problem**: Write a function `extract_uniprot_features_table(html_path: Path, out_csv: Path) -> Path`:
- Read `uniprot_sample.html`.
- Parse the table with `id="features-table"` (or within the features section).
- Extract columns: `'Feature'`, `'Type'`, `'Position'`, `'Description'`.
- Save the parsed table as CSV to `out_csv`.
- Return `out_csv`.


- **Input**: Path to uniprot_sample.html and destination out_csv Path
- **Expected Output**: Path to saved features CSV file

In [ ]:
def extract_uniprot_features_table(html_path: Path, out_csv: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_features = OUT_DIR / "uniprot_features.csv"
p_ft = extract_uniprot_features_table(DATA_DIR / "html" / "uniprot_sample.html", out_features)
assert p_ft.exists() and p_ft.stat().st_size > 50
df_ft = pd.read_csv(p_ft)
assert len(df_ft) > 0
print("Ex 5D.4 passed ✔")

### Practice Exercise 5D.5 (★★★) — Polite Web Scraper Simulator with Pagination & Ethics Logging

**Problem**: Write a function `simulate_polite_scraper(page_htmls: list[str], max_pages: int = 3, out_csv: Path = None) -> tuple[pd.DataFrame, dict]`:
- Takes a list of HTML strings representing paginated search result pages.
- Simulates polite scraping:
  - Enforces rate-limiting pause between pages (`time.sleep(0.05)`).
  - Collects metadata: `'pages_crawled'`, `'items_extracted'`, `'elapsed_seconds'`.
- From each page, parses biomarker rows and consolidates into a single DataFrame.
- If `out_csv` is provided, saves the combined DataFrame.
- Returns `(df_combined, crawl_metadata)`.


- **Input**: List of HTML page strings, max_pages integer, and output CSV Path
- **Expected Output**: Tuple of (consolidated DataFrame, crawl_metadata dictionary)

In [ ]:
def simulate_polite_scraper(page_htmls: list[str], max_pages: int = 3, out_csv: Path = None) -> tuple[pd.DataFrame, dict]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
sample_html = (DATA_DIR / "html" / "table_sample.html").read_text()
out_crawl = OUT_DIR / "polite_scrape_results.csv"
df_crawl, meta_crawl = simulate_polite_scraper([sample_html, sample_html], max_pages=2, out_csv=out_crawl)
assert meta_crawl["pages_crawled"] == 2
assert meta_crawl["items_extracted"] > 0
assert meta_crawl["elapsed_seconds"] >= 0.09
assert out_crawl.exists()
print("Ex 5D.5 passed ✔")

## 6. Summary & Cheat Sheet

Web scraping is a valuable extraction tool when APIs are absent. By establishing an API-first approach, adhering to robots.txt, throttling request rates, and navigating the DOM tree with BeautifulSoup, computational biologists can ethically extract structured tabular data from unstructured web documents.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Robots Parser | RobotFileParser().can_fetch(agent, url) | Audit scraping permissions | Always check before crawling |
| User-Agent Header | headers={'User-Agent': 'Bot/1.0 (email)'} | Identify bot transparently | Avoid fake browser agents |
| BeautifulSoup Init | BeautifulSoup(html, 'html.parser') | Build DOM parse tree | Robust to messy HTML |
| Find Single Element | soup.find('tag', class_='name') | Extract first occurrence | Returns Tag or None |
| Find All Elements | soup.find_all('tag', ...) | Extract all matching occurrences | Returns list of Tags |
| CSS Selector | soup.select('div > article[data-pmid]') | Hierarchical element query | Mirrors browser selectors |
| Text Extraction | tag.get_text(strip=True) | Clean inner text content | Removes whitespace padding |
| Table Parsing | pd.read_html(html) or tr/td loop | Tabular data extraction | Convert numeric types |

## 7. Further Reading & Resources

- [Beautiful Soup 4 Official Documentation](https://beautiful-soup-4.readthedocs.io/en/latest/): Comprehensive guide to DOM traversal and filtering.
- [Python urllib.robotparser Documentation](https://docs.python.org/3/library/urllib.robotparser.html): Official Python documentation for robots.txt parsing.
- [The Web Scraping Club: Ethics & Politeness](https://thewebscraping.club/): Industry guidelines on respectful data harvesting and rate limits.